# 📊 Phase 5: Exploratory Data Analysis (EDA)
**Project:** E-Commerce Sales Analytics Portfolio Project
**Dataset:** `data/cleaned/superstore_cleaned.csv`
**Objective:** Conduct rigorous statistical and commercial exploratory data analysis across 11 key analytical dimensions to evaluate revenue dynamics, profitability drivers, pricing elasticity, and operational logistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load cleaned dataset
CLEANED_DATA_PATH = os.path.join('..', 'data', 'cleaned', 'superstore_cleaned.csv')
df = pd.read_csv(CLEANED_DATA_PATH, dtype={'postal_code': str})
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])
print(f'Validated Cleaned Dataset Loaded: {df.shape[0]:,} records, {df.shape[1]} columns')

## 1. Executive KPIs
**Formulas:**
- **Total Sales:** $\sum(\text{sales})$
- **Total Profit:** $\sum(\text{profit})$
- **Total Quantity:** $\sum(\text{quantity})$
- **Total Orders:** $\text{DistinctCount}(\text{order\_id})$
- **Average Order Value (AOV):** $\frac{\text{Total Sales}}{\text{Total Orders}}$
- **Profit Margin %:** $\frac{\text{Total Profit}}{\text{Total Sales}} \times 100$
- **Average Discount Rate:** $\text{Mean}(\text{discount}) \times 100$
- **Average Shipping Days:** $\text{Mean}(\text{shipping\_days})$

In [ ]:
total_sales = df['sales'].sum()
total_profit = df['profit'].sum()
total_quantity = df['quantity'].sum()
total_orders = df['order_id'].nunique()
aov = total_sales / total_orders
profit_margin = (total_profit / total_sales) * 100
avg_discount = df['discount'].mean() * 100
avg_shipping_days = df['shipping_days'].mean()

kpi_df = pd.DataFrame({
    'KPI Metric': ['Total Revenue', 'Total Net Profit', 'Profit Margin (%)', 'Total Orders', 'Total Units Sold', 'Average Order Value (AOV)', 'Average Discount Rate (%)', 'Average Shipping Days'],
    'Calculated Value': [f'${total_sales:,.2f}', f'${total_profit:,.2f}', f'{profit_margin:.2f}%', f'{total_orders:,}', f'{total_quantity:,} units', f'${aov:.2f}', f'{avg_discount:.2f}%', f'{avg_shipping_days:.2f} days']
})
kpi_df

## 2. Time & Seasonality Analysis
- Annual Performance and Year-over-Year (YoY) Growth
- Monthly Sales and Profit trajectories
- Seasonality by calendar month across all 4 years

In [ ]:
# Yearly summary
yearly = df.groupby('year').agg({'sales': 'sum', 'profit': 'sum', 'order_id': 'nunique'}).reset_index()
yearly['profit_margin_pct'] = (yearly['profit'] / yearly['sales']) * 100
yearly['yoy_sales_growth_pct'] = yearly['sales'].pct_change() * 100
yearly['yoy_profit_growth_pct'] = yearly['profit'].pct_change() * 100
print('=== Annual Summary ===')
print(yearly)

# Monthly Seasonality
monthly_season = df.groupby('month_number').agg({'sales': 'sum', 'profit': 'sum', 'order_id': 'nunique'}).reset_index()
monthly_season['month_name'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print('\n=== Monthly Seasonality ===')
print(monthly_season[['month_name', 'sales', 'profit', 'order_id']])

## 3. Category & Sub-Category Performance
Evaluating commercial performance across Furniture, Office Supplies, and Technology, including identification of loss-making sub-categories.

In [ ]:
cat_summary = df.groupby('category').agg({'sales': 'sum', 'profit': 'sum', 'quantity': 'sum', 'order_id': 'nunique'}).reset_index()
cat_summary['profit_margin_pct'] = (cat_summary['profit'] / cat_summary['sales']) * 100
cat_summary['sales_share_pct'] = (cat_summary['sales'] / total_sales) * 100
cat_summary['profit_share_pct'] = (cat_summary['profit'] / total_profit) * 100
print('=== Category Performance ===')
print(cat_summary.sort_values('profit', ascending=False))

subcat_summary = df.groupby(['category', 'sub_category']).agg({'sales': 'sum', 'profit': 'sum'}).reset_index()
subcat_summary['profit_margin_pct'] = (subcat_summary['profit'] / subcat_summary['sales']) * 100
print('\n=== Loss-Making Sub-Categories ===')
print(subcat_summary[subcat_summary['profit'] < 0].sort_values('profit'))

## 4. Product-Level Deep Dive
- Top 10 Products by Sales
- Top 10 Products by Profit
- Bottom 10 Products by Profit (Severe Loss Leaders)

In [ ]:
prod = df.groupby(['product_name', 'category', 'sub_category']).agg({'sales': 'sum', 'profit': 'sum'}).reset_index()
prod['profit_margin_pct'] = (prod['profit'] / prod['sales']) * 100

print('=== Top 5 Products by Sales ===')
print(prod.sort_values('sales', ascending=False).head(5))

print('\n=== Top 5 Products by Profit ===')
print(prod.sort_values('profit', ascending=False).head(5))

print('\n=== Bottom 5 Products by Profit ===')
print(prod.sort_values('profit', ascending=True).head(5))

## 5. Customer & Segment Distribution
- Customer spend concentration (Pareto Analysis)
- Performance by Customer Segment (Consumer, Corporate, Home Office)

In [ ]:
cust = df.groupby(['customer_id', 'customer_name']).agg({'sales': 'sum', 'profit': 'sum', 'order_id': 'nunique'}).reset_index()
top20pct_n = int(len(cust) * 0.20)
top20pct_rev = cust.sort_values('sales', ascending=False).head(top20pct_n)['sales'].sum()
print(f'Total Unique Customers: {len(cust)}')
print(f'Top 20% Customers ({top20pct_n} accounts) Revenue Share: {(top20pct_rev / total_sales) * 100:.2f}% (${top20pct_rev:,.2f})')

seg = df.groupby('segment').agg({'sales': 'sum', 'profit': 'sum', 'order_id': 'nunique', 'quantity': 'sum'}).reset_index()
seg['profit_margin_pct'] = (seg['profit'] / seg['sales']) * 100
print('\n=== Customer Segment Breakdown ===')
print(seg)

## 6. Regional & State-Level Analysis
- Profitability by US Region
- Top performing states vs bottom 10 loss-making states

In [ ]:
reg = df.groupby('region').agg({'sales': 'sum', 'profit': 'sum', 'order_id': 'nunique'}).reset_index()
reg['profit_margin_pct'] = (reg['profit'] / reg['sales']) * 100
print('=== Regional Performance ===')
print(reg.sort_values('profit', ascending=False))

state_summary = df.groupby(['state', 'region']).agg({'sales': 'sum', 'profit': 'sum', 'discount': 'mean'}).reset_index()
state_summary['profit_margin_pct'] = (state_summary['profit'] / state_summary['sales']) * 100
state_summary['avg_discount_pct'] = state_summary['discount'] * 100
print('\n=== Bottom 5 Deficit States ===')
print(state_summary.sort_values('profit', ascending=True).head(5))

## 7. Discount Sensitivity & Correlation Analysis
Evaluating the empirical relationship between discount percentages and net profit margins.

In [ ]:
disc = df.groupby('discount').agg({'sales': ['count', 'sum'], 'profit': 'sum'}).reset_index()
disc.columns = ['discount', 'order_count', 'total_sales', 'total_profit']
disc['profit_margin_pct'] = (disc['total_profit'] / disc['total_sales']) * 100
print('=== Profitability by Exact Discount Rate ===')
print(disc)

print('\n=== Correlation Matrix ===')
print(df[['sales', 'quantity', 'discount', 'profit', 'shipping_days']].corr())

## 8. Shipping & Fulfillment Analysis
Transit duration metrics across Ship Modes.

In [ ]:
ship = df.groupby('ship_mode').agg({'shipping_days': ['mean', 'min', 'max'], 'sales': 'sum', 'profit': 'sum', 'order_id': 'nunique'}).reset_index()
ship.columns = ['ship_mode', 'avg_shipping_days', 'min_shipping_days', 'max_shipping_days', 'total_sales', 'total_profit', 'order_count']
ship['profit_margin_pct'] = (ship['total_profit'] / ship['total_sales']) * 100
print('=== Shipping Performance ===')
print(ship.sort_values('avg_shipping_days'))